# S2.12 — Shuffle Partitions
**Date completed:** September 2026  
**Status:** In Progress  
**Key config:** spark.sql.shuffle.partitions — most important tuning setting

In [0]:
# ============================================================
# Cell 2 — Shuffle Partitions: Proving the impact
#
# Goal:
# 1. See current shuffle partition setting
# 2. Compare performance with different settings
# 3. Understand why auto mode is default in 2026
#
# Key config: spark.sql.shuffle.partitions
# ============================================================

import pyspark.sql.functions as F
import time

# ------------------------------------------------------------
# Step 1 — Check current shuffle partition setting
# ------------------------------------------------------------
print("=== SHUFFLE PARTITIONS CONFIGURATION ===")
print()

current_setting = spark.conf.get("spark.sql.shuffle.partitions")
print(f"Current setting: spark.sql.shuffle.partitions = {current_setting}")
print()

# ------------------------------------------------------------
# Step 2 — Create test dataset
# ------------------------------------------------------------
df = spark.range(0, 5000000)
df = df.withColumn("city",
    F.when(F.col("id") % 4 == 0, "Delhi")
    .when(F.col("id") % 4 == 1, "Mumbai")
    .when(F.col("id") % 4 == 2, "Pune")
    .otherwise("Chennai"))

print("Dataset: 5M rows, 4 cities")
print()

# ------------------------------------------------------------
# Step 3 — Test with too many shuffle partitions (200)
# ------------------------------------------------------------
print("--- Test 1: 200 shuffle partitions (legacy default) ---")
spark.conf.set("spark.sql.shuffle.partitions", "200")
start = time.time()
result = df.groupBy("city").agg(F.count("*").alias("count"))
result.show()
end = time.time()
time_200 = round(end - start, 4)
print(f"Time with 200 shuffle partitions: {time_200} seconds")
print()

# ------------------------------------------------------------
# Step 4 — Test with optimal shuffle partitions (8)
# ------------------------------------------------------------
print("--- Test 2: 8 shuffle partitions (matched to cores) ---")
spark.conf.set("spark.sql.shuffle.partitions", "8")
start = time.time()
result = df.groupBy("city").agg(F.count("*").alias("count"))
result.show()
end = time.time()
time_8 = round(end - start, 4)
print(f"Time with 8 shuffle partitions: {time_8} seconds")
print()

# ------------------------------------------------------------
# Step 5 — Reset to auto (best practice)
# ------------------------------------------------------------
spark.conf.set("spark.sql.shuffle.partitions", "auto")
print("--- Reset to auto (Databricks 2026 best practice) ---")
print(f"Setting restored: {spark.conf.get('spark.sql.shuffle.partitions')}")
print()

# ------------------------------------------------------------
# Step 6 — Summary
# ------------------------------------------------------------
print("=== COMPARISON ===")
print(f"200 partitions : {time_200} seconds")
print(f"8 partitions   : {time_8} seconds")
print(f"Improvement    : {round(time_200 - time_8, 4)} seconds saved")
print()
print("Rule: Match shuffle partitions to your data size")
print("      Auto mode handles this intelligently in 2026")

In [0]:
# ============================================================
# S2.12 — Cell 3: Shuffle Partitions Experiment
#
# Goal:
# 1. Compare 200, 8, and auto shuffle partition settings.
# 2. Understand the performance impact.
# 3. Preserve the original session configuration.
# ============================================================

import pyspark.sql.functions as F
import time

print("=== SHUFFLE PARTITIONS EXPERIMENT ===")

# ------------------------------------------------------------
# Step 1 — Save original configuration
# ------------------------------------------------------------

original_setting = spark.conf.get(
    "spark.sql.shuffle.partitions"
)

print(f"Original setting: {original_setting}")


# ------------------------------------------------------------
# Step 2 — Create dataset with 10,000 distinct groups
# ------------------------------------------------------------

df = spark.range(0, 5000000)

df = df.withColumn(
    "group_id",
    (F.col("id") % 10000).cast("int")
)

print("Dataset: 5M rows and 10,000 groups")


# ------------------------------------------------------------
# Step 3 — Execute same aggregation with different settings
#
# collect() materializes all 10,000 result groups.
# ------------------------------------------------------------

results = {}

try:

    for setting in ["200", "8", "auto"]:

        print(f"\n--- Testing: {setting} ---")

        spark.conf.set(
            "spark.sql.shuffle.partitions",
            setting
        )

        start = time.perf_counter()

        result = df.groupBy("group_id").agg(
            F.count("*").alias("order_count")
        )

        rows = result.collect()

        elapsed = round(
            time.perf_counter() - start,
            4
        )

        results[setting] = elapsed

        # Validate that every execution produced the same result
        total_rows = sum(
            row["order_count"] for row in rows
        )

        print(f"Groups returned: {len(rows)}")
        print(f"Total records: {total_rows}")
        print(f"Execution time: {elapsed} seconds")


finally:

    # --------------------------------------------------------
    # Step 4 — Always restore original configuration
    # --------------------------------------------------------

    spark.conf.set(
        "spark.sql.shuffle.partitions",
        original_setting
    )

    print(f"\nRestored setting: {original_setting}")


# ------------------------------------------------------------
# Step 5 — Display comparison
# ------------------------------------------------------------

print("\n=== FINAL COMPARISON ===")

for setting, elapsed in results.items():

    print(f"{setting:>5} : {elapsed} seconds")

## Key Takeaways — S2.12 Shuffle Partitions
## The Most Important Spark Config
## 1. What are Shuffle Partitions?
Configuration:

`spark.sql.shuffle.partitions`

Controls the initial number of partitions used when Spark redistributes
intermediate data during shuffle operations.

Common operations that may cause shuffle:
- groupBy()
- join()
- distinct()
- orderBy()

Note: Not every join necessarily requires a shuffle. Spark may use
optimisations such as broadcast joins.

---

## 2. Three Configuration Modes

| Mode | Setting | Explanation |
|------|---------|-------------|
| Standard Apache Spark default | 200 | Default configuration, not universally optimal |
| Manual tuning | Custom integer | Used when performance testing justifies an explicit value |
| Databricks Auto Optimized Shuffle | auto | Databricks determines partition count automatically |

Our Databricks Serverless environment originally used `auto`.

---

## 3. Why Can 200 Partitions Be Inefficient?

Imagine only 1,000 intermediate records distributed evenly
across 200 partitions.

1,000 / 200 = 5 records per partition.

Potential problems:
- Too many small tasks.
- Task scheduling overhead.
- Inefficient resource utilisation.

However, configuring 200 does not guarantee 200 non-empty tasks.
Adaptive Query Execution (AQE) can combine shuffle partitions.

---

## 4. Our Practical Experiment

Dataset:
- 5,000,000 rows
- 10,000 grouping keys

| Shuffle setting | Execution time |
|-----------------|---------------:|
| 200 | 0.7905 seconds |
| 8 | 0.4227 seconds |
| auto | 0.4856 seconds |

All three executions returned:
- 10,000 groups
- 5,000,000 total records

Observation:
8 was the fastest in this particular test.

Conclusion:
More shuffle partitions do not automatically improve performance.
The appropriate count depends on the workload.

These are single-run measurements, not proof that 8 is universally optimal.

---

## 5. How Do We Estimate Shuffle Partitions?

A useful initial sizing approach:

Estimated partitions =
Estimated shuffle data size / Target bytes per partition

Example:

Shuffle data = 8 GB
Illustrative target = 128 MB

Estimated partitions = 64

Important:
128 MB is a starting heuristic, NOT a guaranteed optimal value.

Use the estimated SHUFFLE data volume, not automatically the total
original input table size.

Consider:
- Available CPU cores
- Number of concurrent tasks
- Data distribution and skew
- Memory and spill
- Shuffle read/write
- Query execution time

The final value should be validated through benchmarking.

---

## 6. Adaptive Query Execution (AQE)

AQE allows Spark to adjust parts of the execution plan at runtime.

It can:
- Combine small shuffle partitions.
- Handle certain skewed join partitions.
- Change eligible join strategies.

Therefore:

Configured shuffle partitions ≠ Always final physical partitions.

Databricks Auto Optimized Shuffle can also select the shuffle
partition count based on the query plan and input size.

---

## 7. Production Best Practice

1. Start with the environment's existing configuration.
2. Run the workload.
3. Review Query Profile.
4. Identify whether shuffle is actually expensive.
5. Investigate skew, spill, data movement and task overhead.
6. Test manual partition counts when justified.
7. Compare repeated execution results.
8. Preserve correctness and restore the original configuration.

Do not blindly change shuffle partitions for every pipeline.

---

## 8. Check and Restore Configuration

```python
# Save original setting before experimentation
original_setting = spark.conf.get("spark.sql.shuffle.partitions")

# Example manual experiment
spark.conf.set("spark.sql.shuffle.partitions", "8")

# After completing the experiment, restore the original setting
spark.conf.set("spark.sql.shuffle.partitions", original_setting)
```

For our Databricks Serverless environment, the original setting was `auto`.

---

## 9. Interview Conclusion

Shuffle partitions determine the initial partitioning of redistributed
intermediate data during Spark SQL shuffle operations.

Too few partitions can reduce parallelism and create large tasks.

Too many partitions can introduce unnecessary scheduling and processing
overhead.

Adaptive Query Execution can adjust shuffle partitioning at runtime.

The partition count should be selected using workload characteristics,
available resources and execution metrics rather than a universal number.

**Key principle: Measure first, tune second.**